# 33-风控系统设计

> 模块 4.2 回测与风控 | 凯利公式 + ATR 动态止损 + 回撤限制

## 学习目标

- 理解凯利公式的数学原理和实际应用中的修正
- 实现 ATR（平均真实波幅）动态止损
- 掌握最大回撤限制作为硬风控
- 对比"无风控 vs 有风控"策略的表现差异
- 在上节课手写回测引擎的基础上插入风控检查点

## 环境依赖

`numpy`、`pandas`、`matplotlib`。将上节课的 `BacktestEngine` 升级为带风控模块的版本。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple
from enum import Enum
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. 风控在量化系统中的位置

上节课我们把回测引擎拆成了五大组件。风控模块插在**撮合之后、成交之前**：

```text
信号 → 订单 → 撮合 → [风控检查] → 账户更新 → 记录净值
                        ↑
                 本节课聚焦这里
```

### 三大风控模块

| 模块 | 职责 | 核心问题 |
|------|------|------|
| **凯利公式** | 仓位大小 | 这次下注应该用多少资金？ |
| **ATR 动态止损** | 单笔亏损上限 | 这笔交易最多亏多少就走？ |
| **最大回撤限制** | 全局风控 | 账户整体回撤到多少就停？ |

这三者回答的是量化交易中最基本的三层风控问题：下多少、亏多少跑、整体亏多少停。

## 2. 凯利公式：最优仓位

### 经典凯利公式

凯利公式回答：已知胜率 $p$ 和盈亏比 $b$，每次应投入多大比例的资金？

$$ f^* = \frac{p \cdot b - (1-p)}{b} $$

- $p$：胜率
- $b$：盈亏比（平均盈利 / 平均亏损的绝对值）
- $f^*$：最优投注比例

### 直觉

- 如果 $p = 0.6$，$b = 2$（赢时赚 2 块，输时亏 1 块）
- $f^* = (0.6 × 2 - 0.4) / 2 = 0.4$，即投 40% 资金

### 量化中的修正

真实交易中盈亏比和胜率是估计值，直接使用全仓凯利风险极高。业界通常使用：
- **半凯利（Half Kelly）**：$f = f^* / 2$
- **四分之一凯利（Quarter Kelly）**：$f = f^* / 4$

本节课使用半凯利。

In [ ]:
class KellyPositionSizer:
    """凯利公式仓位管理"""

    def __init__(self, kelly_fraction: float = 0.5, max_position_pct: float = 0.25):
        """
        Args:
            kelly_fraction: 凯利比例系数（0.5=半凯利）
            max_position_pct: 单次最大仓位上限（防止极端值）
        """
        self.kelly_fraction = kelly_fraction
        self.max_position_pct = max_position_pct

    def compute_kelly(self, win_rate: float, profit_loss_ratio: float) -> float:
        """计算凯利比例"""
        if profit_loss_ratio <= 0 or win_rate <= 0:
            return 0.0
        f_star = (win_rate * profit_loss_ratio - (1 - win_rate)) / profit_loss_ratio
        return max(0.0, f_star)  # 凯利不能为负

    def position_size(self, capital: float, price: float,
                       win_rate: float, profit_loss_ratio: float) -> int:
        """计算应买入的股数"""
        kelly_f = self.compute_kelly(win_rate, profit_loss_ratio)
        adjusted_f = min(kelly_f * self.kelly_fraction, self.max_position_pct)

        if adjusted_f <= 0:
            return 0

        amount = capital * adjusted_f
        shares = int(amount / price)
        # 至少 100 股（A 股 1 手），但这里按通用处理
        return max(shares, 1) if shares > 0 else 0

    @staticmethod
    def estimate_metrics(trade_records: List[Dict]) -> Tuple[float, float]:
        """从历史交易记录估计胜率和盈亏比"""
        if not trade_records:
            # 默认值：不能为零
            return 0.5, 1.5

        # 按成交对计算盈亏（简化：相邻的买-卖配对）
        pnl_list = []
        i = 0
        while i < len(trade_records):
            buy = trade_records[i]
            if buy["side"] != "buy":
                i += 1
                continue
            # 找对应的卖出
            for j in range(i + 1, len(trade_records)):
                if trade_records[j]["side"] == "sell" and trade_records[j]["quantity"] == buy["quantity"]:
                    pnl = (trade_records[j]["price"] - buy["price"]) * buy["quantity"]
                    pnl -= buy["fee"] + trade_records[j]["fee"]
                    pnl_list.append(pnl)
                    i = j
                    break
            i += 1

        if not pnl_list:
            return 0.5, 1.5

        wins = [p for p in pnl_list if p > 0]
        losses = [p for p in pnl_list if p < 0]
        win_rate = len(wins) / len(pnl_list) if pnl_list else 0.5
        avg_win = np.mean(wins) if wins else 1
        avg_loss = abs(np.mean(losses)) if losses else 1
        profit_loss_ratio = avg_win / avg_loss if avg_loss > 0 else 2.0

        return win_rate, profit_loss_ratio

# 演示
kelly = KellyPositionSizer(kelly_fraction=0.5)
shares = kelly.position_size(capital=1_000_000, price=10.0, win_rate=0.6, profit_loss_ratio=2.0)
print(f"资本 100万，股价 10，胜率 60%，盈亏比 2.0")
print(f"凯利比例: {kelly.compute_kelly(0.6, 2.0):.2%}")
print(f"半凯利仓位: {shares} 股 = {shares * 10:.0f} 元 = {shares * 10 / 1_000_000:.2%} 资金")

## 3. ATR 动态止损

固定止损（如 -5%）忽略了市场波动率的变化。波动大时容易被震出去，波动小时止损又太宽。

ATR（Average True Range）根据近期波动率自适应调整止损距离：

$$ \text{止损价} = \text{入场价} - N \times \text{ATR} $$

- ATR 越高（市场越剧烈），止损越宽
- ATR 越低（市场越平静），止损越紧

常用 N = 2 倍 ATR。

In [ ]:
class ATRStopLoss:
    """ATR 动态止损"""

    def __init__(self, atr_period: int = 14, atr_multiplier: float = 2.0):
        self.atr_period = atr_period
        self.atr_multiplier = atr_multiplier

    def compute_atr(self, high: np.ndarray, low: np.ndarray, close: np.ndarray) -> np.ndarray:
        """计算 ATR 序列

        True Range = max(high-low, |high-prev_close|, |low-prev_close|)
        ATR = TR 的 N 日移动平均
        """
        n = len(close)
        prev_close = np.roll(close, 1)
        prev_close[0] = close[0]

        tr1 = high - low
        tr2 = np.abs(high - prev_close)
        tr3 = np.abs(low - prev_close)
        true_range = np.maximum(np.maximum(tr1, tr2), tr3)

        # 指数移动平均（EMA 比 SMA 更平滑）
        atr = np.zeros(n)
        atr[0] = true_range[0]
        alpha = 2.0 / (self.atr_period + 1)
        for i in range(1, n):
            atr[i] = alpha * true_range[i] + (1 - alpha) * atr[i - 1]

        return atr

    def stop_price(self, entry_price: float, atr_value: float, side: str = "long") -> float:
        """计算止损价格

        Args:
            side: 'long' 多仓用下轨，'short' 空仓用上轨
        """
        band = self.atr_multiplier * atr_value
        if side == "long":
            return entry_price - band
        else:
            return entry_price + band

    def is_stopped(self, current_low: float, stop_loss_price: float,
                    side: str = "long") -> bool:
        """检查是否触发止损"""
        if side == "long":
            return current_low <= stop_loss_price
        else:
            return current_low >= stop_loss_price  # 这里对空仓用 high

# 演示：用模拟价格计算 ATR
np.random.seed(42)
n = 200
price = 100 * np.exp(np.cumsum(np.random.randn(n) * 0.015))
high = price * (1 + np.abs(np.random.randn(n) * 0.01))
low = price * (1 - np.abs(np.random.randn(n) * 0.01))

atr_calc = ATRStopLoss(atr_period=14, atr_multiplier=2.0)
atr_series = atr_calc.compute_atr(high, low, price)

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(price, linewidth=1, label="收盘价")
axes[0].set_title("模拟价格")
axes[0].legend()
axes[1].plot(atr_series, linewidth=1, color="coral")
axes[1].set_title(f"ATR({atr_calc.atr_period})")
axes[1].set_ylabel("ATR")
plt.tight_layout()
plt.show()

print(f"ATR 范围: {atr_series.min():.4f} ~ {atr_series.max():.4f}")
print(f"当前 ATR: {atr_series[-1]:.4f}")
print(f"若入场价={price[-1]:.2f}，止损价={atr_calc.stop_price(price[-1], atr_series[-1], 'long'):.2f}")
print(f"止损距离: {atr_calc.atr_multiplier * atr_series[-1]:.2f} = {(atr_calc.atr_multiplier * atr_series[-1] / price[-1]):.2%} 当前价的")

## 4. 最大回撤限制

前两个风控管单笔交易，最大回撤限制管**整个账户**。

规则：当账户从峰值回撤超过阈值（如 20%），停止所有新交易，直到回撤恢复到某个水平。

这模拟了真实资管中的"风控线"——触及必须暂停交易的纪律。

In [ ]:
class DrawdownLimiter:
    """最大回撤限制"""

    def __init__(self, max_drawdown_pct: float = 0.20, recovery_pct: float = 0.10):
        """
        Args:
            max_drawdown_pct: 触发风控的回撤阈值（默认 20%）
            recovery_pct: 恢复到该回撤水平以下才允许重新交易（默认 10%）
        """
        self.max_drawdown_pct = max_drawdown_pct
        self.recovery_pct = recovery_pct
        self.peak_nav = None
        self.trading_halted = False

    def update(self, current_nav: float) -> bool:
        """更新状态，返回是否允许交易"""
        if self.peak_nav is None:
            self.peak_nav = current_nav

        self.peak_nav = max(self.peak_nav, current_nav)
        current_dd = (current_nav - self.peak_nav) / self.peak_nav

        if self.trading_halted:
            # 检查是否恢复
            if abs(current_dd) < self.recovery_pct:
                self.trading_halted = False
            return False
        else:
            if abs(current_dd) >= self.max_drawdown_pct:
                self.trading_halted = True
                return False
            return True

# 演示
dd_limiter = DrawdownLimiter(max_drawdown_pct=0.20, recovery_pct=0.10)
nav_series = np.array([100, 105, 95, 90, 78, 80, 82, 92, 95])
for i, nav in enumerate(nav_series):
    allowed = dd_limiter.update(nav)
    dd = (nav - dd_limiter.peak_nav) / dd_limiter.peak_nav
    status = "🚫 停" if dd_limiter.trading_halted else ("✅" if allowed else "-")
    print(f"Day {i}: NAV={nav:.0f} DD={dd:.2%} Peak={dd_limiter.peak_nav:.0f} {status}")

## 5. 整合：带风控的回测引擎

将三个风控模块集成到上节课的回测引擎中。关键改动在主循环里：

```text
for each bar:
    1. 更新最大回撤限制
    2. 如果被停 → skip 交易
    3. 策略生成信号
    4. 用凯利公式计算仓位
    5. 撮合
    6. ATR 止损检查（持仓）
    7. 账户更新 + 快照
```

In [ ]:
class OrderType(Enum):
    MARKET = "market"
    LIMIT = "limit"

class OrderSide(Enum):
    BUY = "buy"
    SELL = "sell"

@dataclass
class Order:
    order_id: int
    symbol: str
    order_type: OrderType
    side: OrderSide
    quantity: int
    limit_price: Optional[float] = None
    timestamp: Optional[int] = None

@dataclass
class Fill:
    order_id: int
    symbol: str
    side: OrderSide
    quantity: int
    price: float
    fee: float
    timestamp: int

@dataclass
class MarketData:
    timestamp: int
    open: float
    high: float
    low: float
    close: float
    volume: int

@dataclass
class Account:
    initial_cash: float = 1_000_000
    cash: float = None
    positions: Dict[str, int] = field(default_factory=dict)
    position_records: List[Dict] = field(default_factory=list)
    trade_records: List[Dict] = field(default_factory=list)

    def __post_init__(self):
        if self.cash is None:
            self.cash = self.initial_cash

    def can_afford(self, order: Order, price: float) -> bool:
        cost = price * order.quantity * 1.0003
        return self.cash >= cost

    def has_position(self, symbol: str, quantity: int) -> bool:
        return self.positions.get(symbol, 0) >= quantity

    def apply_fill(self, fill: Fill, current_price: float):
        symbol = fill.symbol
        if fill.side == OrderSide.BUY:
            cost = fill.price * fill.quantity + fill.fee
            self.cash -= cost
            self.positions[symbol] = self.positions.get(symbol, 0) + fill.quantity
        else:
            revenue = fill.price * fill.quantity - fill.fee
            self.cash += revenue
            self.positions[symbol] = self.positions.get(symbol, 0) - fill.quantity
            if self.positions[symbol] <= 0:
                del self.positions[symbol]
        self.trade_records.append({
            "timestamp": fill.timestamp,
            "symbol": fill.symbol,
            "side": fill.side.value,
            "quantity": fill.quantity,
            "price": fill.price,
            "fee": fill.fee,
        })

    def total_value(self, price_map: Dict[str, float]) -> float:
        pos_val = sum(qty * price_map.get(sym, 0) for sym, qty in self.positions.items())
        return self.cash + pos_val

    def snapshot(self, timestamp: int, price_map: Dict[str, float]):
        tv = self.total_value(price_map)
        self.position_records.append({
            "timestamp": timestamp,
            "cash": self.cash,
            "positions": dict(self.positions),
            "total_value": tv,
        })

In [ ]:
class MatchingEngine:
    """撮合引擎（同上节课）"""
    def __init__(self, slippage: float = 0.001, fee_rate: float = 0.0003):
        self.slippage = slippage
        self.fee_rate = fee_rate
        self.fill_id = 0

    def match(self, order: Order, market_data: MarketData) -> Optional[Fill]:
        if order.quantity <= 0:
            return None
        if order.order_type == OrderType.MARKET:
            fill_price = market_data.close
        elif order.order_type == OrderType.LIMIT:
            if order.side == OrderSide.BUY:
                if market_data.low <= order.limit_price:
                    fill_price = min(order.limit_price, market_data.close)
                else:
                    return None
            else:
                if market_data.high >= order.limit_price:
                    fill_price = max(order.limit_price, market_data.close)
                else:
                    return None
        else:
            return None

        if order.side == OrderSide.BUY:
            fill_price *= (1 + self.slippage)
        else:
            fill_price *= (1 - self.slippage)

        fee = fill_price * order.quantity * self.fee_rate
        self.fill_id += 1
        return Fill(order.order_id, order.symbol, order.side, order.quantity,
                    fill_price, fee, market_data.timestamp)


class RiskManagedBacktestEngine:
    """带风控模块的回测引擎"""

    def __init__(self, initial_cash=1_000_000, slippage=0.001, fee_rate=0.0003,
                 use_kelly=True, use_atr=True, use_dd_limit=True):
        self.account = Account(initial_cash=initial_cash)
        self.matcher = MatchingEngine(slippage=slippage, fee_rate=fee_rate)
        self.order_counter = 0

        # 风控开关
        self.use_kelly = use_kelly
        self.use_atr = use_atr
        self.use_dd_limit = use_dd_limit

        # 风控模块
        self.kelly = KellyPositionSizer(kelly_fraction=0.5, max_position_pct=0.25)
        self.atr = ATRStopLoss(atr_period=14, atr_multiplier=2.0)
        self.dd_limiter = DrawdownLimiter(max_drawdown_pct=0.20, recovery_pct=0.10)

        # ATR 止损追踪
        self.active_stops: Dict[str, float] = {}  # symbol → stop_price
        self.atr_series: Optional[np.ndarray] = None

    def create_order(self, symbol, side, quantity, order_type=OrderType.MARKET,
                     limit_price=None):
        self.order_counter += 1
        return Order(self.order_counter, symbol, order_type, side, quantity, limit_price)

    def run(self, price_df: pd.DataFrame, signal_fn) -> Dict:
        n = len(price_df)

        # 预先计算 ATR
        if self.use_atr:
            self.atr_series = self.atr.compute_atr(
                price_df["high"].values,
                price_df["low"].values,
                price_df["close"].values,
            )

        for i in range(n):
            row = price_df.iloc[i]
            md = MarketData(i, row["open"], row["high"], row["low"], row["close"], row.get("volume", 0))
            price_map = {"000001": md.close}

            # === 风控 1：最大回撤限制 ===
            current_nav = self.account.total_value(price_map)
            if self.use_dd_limit:
                trading_allowed = self.dd_limiter.update(current_nav)
                if not trading_allowed:
                    self.account.snapshot(i, price_map)
                    continue

            # === 风控 2：ATR 止损检查（已有持仓） ===
            if self.use_atr and self.atr_series is not None:
                for sym, qty in list(self.account.positions.items()):
                    if sym in self.active_stops:
                        stop_price = self.active_stops[sym]
                        if md.low <= stop_price:
                            # 触发止损：市价卖出全部持仓
                            stop_order = self.create_order(sym, OrderSide.SELL, qty)
                            stop_fill = self.matcher.match(stop_order, md)
                            if stop_fill:
                                self.account.apply_fill(stop_fill, md.close)
                                del self.active_stops[sym]

            # === 策略信号 ===
            orders = signal_fn(i, price_df, self.account)

            for order in orders:
                # === 风控 3：凯利仓位调整 ===
                if self.use_kelly and order.side == OrderSide.BUY:
                    wr, plr = KellyPositionSizer.estimate_metrics(self.account.trade_records)
                    opt_qty = self.kelly.position_size(
                        self.account.cash, md.close, wr, plr
                    )
                    if opt_qty > 0 and opt_qty < order.quantity:
                        order.quantity = opt_qty

                # 撮合
                fill = self.matcher.match(order, md)
                if not fill:
                    continue

                # 资金/持仓检查
                if fill.side == OrderSide.BUY:
                    if not self.account.can_afford(order, fill.price):
                        continue
                else:
                    if not self.account.has_position(order.symbol, order.quantity):
                        continue

                self.account.apply_fill(fill, md.close)

                # ATR 止损设置（买入后）
                if self.use_atr and self.atr_series is not None and fill.side == OrderSide.BUY:
                    if i < len(self.atr_series):
                        atr_val = self.atr_series[i]
                        stop = self.atr.stop_price(fill.price, atr_val, "long")
                        self.active_stops[fill.symbol] = stop

            # 记录快照
            self.account.snapshot(i, price_map)

        return self._compute_performance()

    def _compute_performance(self) -> Dict:
        records = self.account.position_records
        navs = np.array([r["total_value"] for r in records])
        returns = np.diff(navs) / navs[:-1]

        if len(returns) == 0:
            return {"error": "无收益数据"}

        total_return = navs[-1] / navs[0] - 1
        annual_return = (1 + total_return) ** (252 / len(returns)) - 1
        annual_vol = np.std(returns) * np.sqrt(252)
        sharpe = (annual_return - 0.02) / annual_vol if annual_vol > 0 else 0
        peak = np.maximum.accumulate(navs)
        drawdowns = (navs - peak) / peak
        max_dd = drawdowns.min()
        calmar = annual_return / abs(max_dd) if max_dd != 0 else 0
        win_rate = (returns > 0).mean()
        gains = returns[returns > 0]
        losses = returns[returns < 0]
        profit_loss = gains.mean() / abs(losses.mean()) if len(losses) > 0 else float("inf")

        return {
            "累计收益": total_return,
            "年化收益": annual_return,
            "年化波动": annual_vol,
            "Sharpe": sharpe,
            "最大回撤": max_dd,
            "Calmar": calmar,
            "胜率": win_rate,
            "盈亏比": profit_loss,
            "nav": navs,
            "returns": returns,
            "drawdowns": drawdowns,
            "trades": self.account.trade_records,
            "stopped_by_dd": self.dd_limiter.trading_halted,
            "stop_events": sum(1 for t in self.account.trade_records if t.get("reason") == "stop_loss"),
        }

## 6. 实战：无风控 vs 有风控 对比

使用同一段模拟价格数据，同一策略，跑两组实验：
- A 组：无风控
- B 组：开启凯利 + ATR + 回撤限制

In [ ]:
def generate_ohlcv_data(n=500):
    """生成含一段明显熊市的 OHLCV 数据"""
    np.random.seed(RANDOM_SEED)
    dates = pd.date_range("2020-01-01", periods=n, freq="B")

    # 前半段牛市，后半段熊市
    mu = np.where(np.arange(n) < n * 0.55, 0.001, -0.0015)
    sigma = 0.018
    noise = np.random.randn(n) * sigma
    log_returns = mu + noise
    price = 100 * np.exp(np.cumsum(log_returns))

    df = pd.DataFrame({"date": dates, "close": price})
    daily_range = np.abs(np.random.randn(n) * 0.012) + 0.005
    df["open"] = df["close"].shift(1).fillna(price[0])
    df["high"] = np.maximum(df["open"], df["close"]) * (1 + daily_range / 2)
    df["low"] = np.minimum(df["open"], df["close"]) * (1 - daily_range / 2)
    df["volume"] = np.random.randint(50000, 200000, n)
    df.set_index("date", inplace=True)
    return df

price_df = generate_ohlcv_data(500)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(price_df.index, price_df["close"], linewidth=1)
ax.set_title("模拟价格（前半段牛市 + 后半段熊市）")
ax.set_ylabel("价格")
plt.show()

In [ ]:
def ma_cross_signal(timestamp, price_df, account):
    """均线交叉策略（同 32 课），每次固定 1000 股"""
    symbol = "000001"
    base_qty = 1000

    if timestamp < 21:
        return []

    closes = price_df["close"].values[:timestamp + 1]
    ma5 = np.mean(closes[-5:])
    ma20 = np.mean(closes[-20:])

    closes_prev = price_df["close"].values[:timestamp]
    if len(closes_prev) < 20:
        return []
    ma5_prev = np.mean(closes_prev[-5:])
    ma20_prev = np.mean(closes_prev[-20:])

    orders = []
    # 金叉 → 买入
    if ma5_prev <= ma20_prev and ma5 > ma20:
        close_price = price_df["close"].values[timestamp]
        if account.can_afford(Order(0, symbol, OrderType.MARKET, OrderSide.BUY, base_qty), close_price):
            orders.append(Order(-1, symbol, OrderType.MARKET, OrderSide.BUY, base_qty, timestamp=timestamp))
    # 死叉 → 卖出
    if ma5_prev >= ma20_prev and ma5 < ma20:
        if account.has_position(symbol, base_qty):
            orders.append(Order(-1, symbol, OrderType.MARKET, OrderSide.SELL, base_qty, timestamp=timestamp))
    return orders

# 无风控
engine_no_risk = RiskManagedBacktestEngine(
    use_kelly=False, use_atr=False, use_dd_limit=False
)
result_no_risk = engine_no_risk.run(price_df, signal_fn=ma_cross_signal)

# 有风控
engine_with_risk = RiskManagedBacktestEngine(
    use_kelly=True, use_atr=True, use_dd_limit=True
)
result_with_risk = engine_with_risk.run(price_df, signal_fn=ma_cross_signal)

# 基准（买入持有）
benchmark_nav = 1_000_000 * (price_df["close"].values / price_df["close"].values[0])

print("=" * 55)
print(f"{'指标':<12s} {'无风控':>12s} {'有风控':>12s} {'基准':>12s}")
print("=" * 55)
for key in ["累计收益", "年化收益", "年化波动", "Sharpe", "最大回撤", "Calmar"]:
    v_no = result_no_risk[key]
    v_with = result_with_risk[key]
    print(f"{key:<12s} {v_no:>11.2%} {v_with:>11.2%}", end="")
    if key == "累计收益":
        bm = benchmark_nav[-1] / benchmark_nav[0] - 1
        print(f" {bm:>11.2%}")
    elif key == "最大回撤":
        bm_dd = (benchmark_nav / np.maximum.accumulate(benchmark_nav) - 1).min()
        print(f" {bm_dd:>11.2%}")
    else:
        print()

print(f"{'成交笔数':<12s} {len(result_no_risk['trades']):>11d} {len(result_with_risk['trades']):>11d}")
print(f"{'风控暂停':<12s} {'--':>12s} {str(result_with_risk['stopped_by_dd']):>12s}")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 净值
axes[0].plot(result_no_risk["nav"], label="无风控", linewidth=1.5, color="coral")
axes[0].plot(result_with_risk["nav"], label="有风控(凯利+ATR+DD)", linewidth=1.5, color="steelblue")
axes[0].plot(benchmark_nav, label="买入持有", linewidth=1, linestyle="--", alpha=0.5, color="gray")
axes[0].set_title("净值曲线对比")
axes[0].legend()
axes[0].set_ylabel("净值")

# 回撤
axes[1].fill_between(range(len(result_no_risk["drawdowns"])),
                     result_no_risk["drawdowns"] * 100, 0, alpha=0.3, color="coral", label="无风控")
axes[1].fill_between(range(len(result_with_risk["drawdowns"])),
                     result_with_risk["drawdowns"] * 100, 0, alpha=0.3, color="steelblue", label="有风控")
axes[1].axhline(y=-20, color="red", linestyle="--", linewidth=0.8, alpha=0.5, label="-20% 风控线")
axes[1].set_title("回撤对比 (%)")
axes[1].set_ylabel("回撤")
axes[1].legend()

# 仓位对比（凯利 vs 固定）
pos_no = np.array([r["positions"].get("000001", 0) for r in engine_no_risk.account.position_records])
pos_with = np.array([r["positions"].get("000001", 0) for r in engine_with_risk.account.position_records])
axes[2].plot(pos_no, linewidth=1, color="coral", alpha=0.7, label="无风控(固定1000股)")
axes[2].plot(pos_with, linewidth=1, color="steelblue", alpha=0.7, label="有风控(凯利动态)")
axes[2].set_title("持仓股数对比")
axes[2].set_ylabel("股数")
axes[2].set_xlabel("交易日")
axes[2].legend()

plt.tight_layout()
plt.show()

## 7. 凯利公式的局限

上面看到了凯利的好处（动态调整仓位），但也要知道它的局限：

| 局限 | 说明 |
|------|------|
| **参数估计误差** | 胜率和盈亏比是从历史估计的，未来可能不成立 |
| **非独立投注** | 凯利假设每次下注独立，但交易有序列相关性 |
| **回撤容忍度** | 全仓凯利的回撤可能远超你的心理承受力 |
| **多资产不适用** | 经典凯利是单次投注，多资产需要更复杂的方法 |
| **尾部风险** | 凯利不能防御黑天鹅——它假设收益分布已知 |

这就是为什么实际应用中使用半凯利或四分之一凯利，并配合 ATR 止损和最大回撤限制一起使用。

三者各司其职：
- 凯利管 **"下多少"**
- ATR 管 **"单笔最多亏多少"**
- 回撤限制管 **"整体最多亏多少"**

## 8. 小结

这节课我们：

1. **实现了凯利公式仓位管理**：从历史交易估计胜率和盈亏比，动态调整每次买入数量
2. **实现了 ATR 动态止损**：根据市场波动自动扩展或收缩止损距离
3. **实现了最大回撤限制**：账户回撤超阈值暂停交易，恢复到安全水位再重启
4. **集成到回测引擎**：在上节课的引擎中插入三个风控检查点
5. **对比验证**：同一策略、同一数据，无风控 vs 有风控的绩效差异一目了然

### 核心收获

风控不是"加一个模块"，而是在回测主循环的每个关键节点插入检查。一个好的风控系统= 仓位管理 + 单笔止损 + 全局回撤限制，三者缺一不可。

## 验收清单

- [ ] 能写出凯利公式并解释每个变量的含义
- [ ] 理解为什么实际用半凯利而不是全仓凯利
- [ ] 能手写 ATR 计算和动态止损逻辑
- [ ] 理解最大回撤限制的触发和恢复机制
- [ ] 能在回测主循环中正确插入风控检查点
- [ ] 能对比有无风控的净值曲线、回撤曲线和仓位变化
- [ ] 知道凯利公式的三个主要局限